```bash
docker compose up -d --build postgres minio spark-master spark-worker-1 spark-worker-2 jupyter-workspace spark-history nessie dremio clickhouse
```

```bash
docker compose up -d postgres minio spark-master spark-worker-1 spark-worker-2 jupyter-workspace spark-history nessie dremio clickhouse
```

### MinIO init_buckets


In [1]:
import io
import sys
from minio import Minio

# سنحاول الاتصال بـ localhost:9005 (من خارج Docker) أو minio:9000 (من داخل Docker) تلقائياً
minio_endpoint = None
minio_client = None

for endpoint in ["localhost:9005", "minio:9000"]:
    try:
        client = Minio(
            endpoint,
            access_key="minioadmin",
            secret_key="minioadmin",
            secure=False
        )
        # اختبار الاتصال
        client.list_buckets()
        minio_client = client
        minio_endpoint = endpoint
        print(f"📡 Connected to MinIO successfully at: {endpoint}")
        break
    except Exception:
        continue

if not minio_client:
    print("❌ Failed to connect to MinIO on localhost:9005 or minio:9000.")
    print("Make sure MinIO is running in Docker (docker compose up).")
    sys.exit(1)

# 1. إنشاء الـ Buckets المطلوبة
buckets = ["warehouse", "spark-logs"]

for bucket in buckets:
    if not minio_client.bucket_exists(bucket):
        minio_client.make_bucket(bucket)
        print(f"✅ Created bucket: '{bucket}'")
    else:
        print(f"ℹ️ Bucket '{bucket}' already exists.")

# 2. إنشاء مجلد event-logs/ داخل spark-logs (مطلوب لـ Spark History Server)
try:
    minio_client.put_object("spark-logs", "event-logs/", io.BytesIO(b""), 0)
    print("✅ Created event-logs directory structure in 'spark-logs' bucket.")
except Exception as e:
    print("❌ Failed to create event-logs directory structure:", e)

print("\n🎉 All MinIO buckets are initialized and ready!")


📡 Connected to MinIO successfully at: localhost:9005
ℹ️ Bucket 'warehouse' already exists.
ℹ️ Bucket 'spark-logs' already exists.
✅ Created event-logs directory structure in 'spark-logs' bucket.

🎉 All MinIO buckets are initialized and ready!


In [2]:
import requests
import json
import sys
import time

# إعدادات الاتصال وحساب الـ Admin
DREMIO_HOST = "localhost"
DREMIO_PORT = 9047
USERNAME = "root"
PASSWORD = "sddffsdd125"  # كلمة مرور الـ Admin الخاصة بك

print("🚀 Starting Dremio Configuration Script...\n")

# 0. إنشاء حساب الـ Admin لأول مرة (Bootstrap First User)
bootstrap_url = f"http://{DREMIO_HOST}:{DREMIO_PORT}/apiv2/bootstrap/firstuser"
bootstrap_payload = {
    "userName": USERNAME,
    "firstName": "Admin",
    "lastName": "User",
    "email": "admin@platform.local",
    "password": PASSWORD
}

print(f"0️⃣ Checking if Dremio needs first-user setup...")
try:
    boot_res = requests.put(bootstrap_url, json=bootstrap_payload)
    if boot_res.status_code == 200:
        print("   ✅ First admin user created successfully!")
        time.sleep(2) # انتظار قليل حتى يستوعب السيرفر
    elif boot_res.status_code in (403, 409):
        print("   ℹ️ Admin user already exists. Bypassing setup...")
    else:
        print(f"   ⚠️ Unexpected response: {boot_res.status_code} - {boot_res.text}")
except Exception as e:
    print(f"❌ Failed to reach Dremio: {e}")
    sys.exit(1)


# 1. تسجيل الدخول للحصول على Token
login_url = f"http://{DREMIO_HOST}:{DREMIO_PORT}/apiv2/login"
login_payload = {
    "userName": USERNAME,
    "password": PASSWORD
}

print(f"\n1️⃣ Sending Login Request to: {login_url}")
try:
    login_res = requests.post(login_url, json=login_payload)
    if login_res.status_code != 200:
        print(f"❌ Failed to login. Details: {login_res.text}")
        sys.exit(1)
        
    token = login_res.json().get("token")
    print("✅ Login successful! Token obtained.")
except Exception as e:
    print(f"❌ Connection error during login: {e}")
    sys.exit(1)

# 2. إنشاء مصدر Nessie
if token:
    catalog_url = f"http://{DREMIO_HOST}:{DREMIO_PORT}/api/v3/catalog"
    headers = {
        "Authorization": f"_dremio{token}",
        "Content-Type": "application/json"
    }

    nessie_source = {
        "entityType": "source",
        "name": "nessie",
        "type": "NESSIE",
        "config": {
            "nessieEndpoint": "http://nessie:19120/api/v2",
            "nessieAuthType": "NONE",
            "secure": False,                       
            "credentialType": "ACCESS_KEY",
            "awsAccessKey": "minioadmin",
            "awsAccessSecret": "minioadmin",
            "awsRootPath": "/warehouse",
            "propertyList": [
                {"name": "fs.s3a.endpoint", "value": "minio:9000"},
                {"name": "fs.s3a.path.style.access", "value": "true"},
                {"name": "dremio.s3.compat", "value": "true"},
                {"name": "fs.s3a.connection.ssl.enabled", "value": "false"}
            ]
        }
    }

    print(f"\n2️⃣ Sending Source Creation Request...")
    try:
        res = requests.post(catalog_url, json=nessie_source, headers=headers)
        if res.status_code in (200, 201):
            print("✅ Nessie source connected to Dremio successfully!")
        elif res.status_code == 409:
            print("ℹ️ Nessie source already exists in Dremio.")
        else:
            print(f"❌ Failed to connect (Status {res.status_code}): {res.text}")
    except Exception as e:
        print(f"❌ Connection error during source creation: {e}")


🚀 Starting Dremio Configuration Script...

0️⃣ Checking if Dremio needs first-user setup...
   ⚠️ Unexpected response: 400 - {"errorMessage":"First user can only be created when no user is already registered","context":[],"moreInfo":""}

1️⃣ Sending Login Request to: http://localhost:9047/apiv2/login
✅ Login successful! Token obtained.

2️⃣ Sending Source Creation Request...
ℹ️ Nessie source already exists in Dremio.


### Dremio-NESSIE

In [3]:
import requests
import json
import sys

# إعدادات الاتصال وحساب الـ Admin
DREMIO_HOST = "localhost"
DREMIO_PORT = 9047
USERNAME = "root"
PASSWORD = "sddffsdd125"  # كلمة مرور الـ Admin الخاصة بك

print("🚀 Starting Dremio Configuration Script...")

# 1. تسجيل الدخول للحصول على Token
login_url = f"http://{DREMIO_HOST}:{DREMIO_PORT}/apiv2/login"
login_payload = {
    "userName": USERNAME,
    "password": PASSWORD
}

print(f"\n1️⃣ Sending Login Request to: {login_url}")
print(f"   Payload: {json.dumps({'userName': USERNAME, 'password': '***'})}")

try:
    login_res = requests.post(login_url, json=login_payload)
    print(f"   Response Status Code: {login_res.status_code}")
    
    if login_res.status_code != 200:
        print(f"❌ Failed to login. Details: {login_res.text}")
        sys.exit(1)
        
    token = login_res.json().get("token")
    print("✅ Login successful! Token obtained.")
except Exception as e:
    print(f"❌ Connection error during login: {e}")
    sys.exit(1)

# 2. إنشاء مصدر Nessie
if token:
    catalog_url = f"http://{DREMIO_HOST}:{DREMIO_PORT}/api/v3/catalog"
    headers = {
        "Authorization": f"_dremio{token}",
        "Content-Type": "application/json"
    }

    # تم إضافة "secure": False لتعطيل الـ SSL عند الاتصال بـ MinIO
    nessie_source = {
        "entityType": "source",
        "name": "nessie",
        "type": "NESSIE",
        "config": {
            "nessieEndpoint": "http://nessie:19120/api/v2",
            "nessieAuthType": "NONE",
            "secure": False,                       # تعطيل التشفير (SSL) للاتصال بـ MinIO
            "credentialType": "ACCESS_KEY",
            "awsAccessKey": "minioadmin",
            "awsAccessSecret": "minioadmin",
            "awsRootPath": "/warehouse",
            "propertyList": [
                {"name": "fs.s3a.endpoint", "value": "minio:9000"},
                {"name": "fs.s3a.path.style.access", "value": "true"},
                {"name": "dremio.s3.compat", "value": "true"},
                {"name": "fs.s3a.connection.ssl.enabled", "value": "false"}
            ]
        }
    }

    print(f"\n2️⃣ Sending Source Creation Request to: {catalog_url}")
    print(f"   Headers: {json.dumps({'Content-Type': 'application/json', 'Authorization': '_dremio***'})}")
    print(f"   Payload: {json.dumps(nessie_source, indent=2)}")

    try:
        res = requests.post(catalog_url, json=nessie_source, headers=headers)
        print(f"   Response Status Code: {res.status_code}")
        
        if res.status_code in (200, 201):
            print("✅ Nessie source connected to Dremio successfully!")
        else:
            print(f"❌ Failed to connect (Status {res.status_code}): {res.text}")
    except Exception as e:
        print(f"❌ Connection error during source creation: {e}")


🚀 Starting Dremio Configuration Script...

1️⃣ Sending Login Request to: http://localhost:9047/apiv2/login
   Payload: {"userName": "root", "password": "***"}
   Response Status Code: 200
✅ Login successful! Token obtained.

2️⃣ Sending Source Creation Request to: http://localhost:9047/api/v3/catalog
   Headers: {"Content-Type": "application/json", "Authorization": "_dremio***"}
   Payload: {
  "entityType": "source",
  "name": "nessie",
  "type": "NESSIE",
  "config": {
    "nessieEndpoint": "http://nessie:19120/api/v2",
    "nessieAuthType": "NONE",
    "secure": false,
    "credentialType": "ACCESS_KEY",
    "awsAccessKey": "minioadmin",
    "awsAccessSecret": "minioadmin",
    "awsRootPath": "/warehouse",
    "propertyList": [
      {
        "name": "fs.s3a.endpoint",
        "value": "minio:9000"
      },
      {
        "name": "fs.s3a.path.style.access",
        "value": "true"
      },
      {
        "name": "dremio.s3.compat",
        "value": "true"
      },
      {
       

### clear_data

In [2]:
import io
import sys
import subprocess
import requests
from minio import Minio
from minio.deleteobjects import DeleteObject

print("🧹 Starting Data Cleanup Script...")

# 1. تهيئة الاتصال بـ MinIO
minio_client = None
minio_endpoint = None

for endpoint in ["localhost:9005", "minio:9000"]:
    try:
        client = Minio(
            endpoint,
            access_key="minioadmin",
            secret_key="minioadmin",
            secure=False
        )
        client.list_buckets()
        minio_client = client
        minio_endpoint = endpoint
        print(f"📡 Connected to MinIO at: {endpoint}")
        break
    except Exception:
        continue

if not minio_client:
    print("❌ Failed to connect to MinIO.")
    sys.exit(1)

# 2. مسح الداتا ليك (MinIO Buckets) وإعادة إنشائها
buckets = ["warehouse", "spark-logs"]

print("\n🌊 Cleaning up MinIO Data Lake...")
for bucket in buckets:
    if minio_client.bucket_exists(bucket):
        print(f"   Deleting objects from bucket '{bucket}'...")
        # جلب كل الملفات وحذفها دفعة واحدة
        try:
            objects = minio_client.list_objects(bucket, recursive=True)
            delete_list = [DeleteObject(obj.object_name) for obj in objects]
            if delete_list:
                errors = minio_client.remove_objects(bucket, delete_list)
                for err in errors:
                    print(f"   ❌ Error deleting: {err}")
            
            # حذف البوكيت نفسه
            minio_client.remove_bucket(bucket)
            print(f"   ✅ Removed bucket: '{bucket}'")
        except Exception as e:
            print(f"   ❌ Failed to remove bucket '{bucket}': {e}")
    
    # إعادة إنشاء البوكيت فارغ
    minio_client.make_bucket(bucket)
    print(f"   ✅ Re-created empty bucket: '{bucket}'")

# إعادة إنشاء مجلد event-logs/ المطلوب لـ Spark History
try:
    minio_client.put_object("spark-logs", "event-logs/", io.BytesIO(b""), 0)
    print("   ✅ Created event-logs directory structure in 'spark-logs' bucket.")
except Exception as e:
    print("   ❌ Failed to create event-logs directory:", e)


# 3. مسح الداتا ويرهاوس (ClickHouse Tables)
print("\n🏢 Cleaning up ClickHouse Data Warehouse...")
clickhouse_url = "http://localhost:8123/"
clickhouse_params = {"user": "admin", "password": "admin"}

for db in ["default", "gold"]:
    try:
        # التحقق مما إذا كانت الداتابيز موجودة أولاً
        exists_query = f"EXISTS DATABASE {db}"
        exists_res = requests.post(clickhouse_url, params=clickhouse_params, data=exists_query)
        
        if exists_res.status_code == 200 and exists_res.text.strip() == "1":
            # جلب قائمة الجداول إذا كانت موجودة
            show_query = f"SHOW TABLES FROM {db}"
            res = requests.post(clickhouse_url, params=clickhouse_params, data=show_query)
            if res.status_code == 200:
                tables = [t.strip() for t in res.text.strip().split("\n") if t.strip()]
                if tables:
                    for table in tables:
                        drop_query = f"DROP TABLE {db}.{table}"
                        drop_res = requests.post(clickhouse_url, params=clickhouse_params, data=drop_query)
                        if drop_res.status_code == 200:
                            print(f"   ✅ Dropped ClickHouse table: {db}.{table}")
                        else:
                            print(f"   ❌ Failed to drop table {db}.{table}: {drop_res.text}")
                else:
                    print(f"   ℹ️ ClickHouse database '{db}' is empty (no tables to drop).")
            else:
                print(f"   ❌ Failed to list tables for DB '{db}': {res.text}")
        else:
            print(f"   ℹ️ ClickHouse database '{db}' does not exist yet (skipping).")
    except Exception as e:
        print(f"   ❌ ClickHouse DB '{db}' cleanup error: {e}")


# 4. تصفير Nessie Catalog (عن طريق إعادة تشغيل الحاوية كونها تحفظ البيانات في الذاكرة المؤقتة In-Memory)
print("\n🌳 Resetting Nessie Catalog...")
try:
    # إعادة تشغيل حاوية نسي
    subprocess.run(["docker", "compose", "restart", "nessie"], check=True)
    print("   ✅ Nessie Catalog restarted and reset to clean state.")
except Exception as e:
    print(f"   ❌ Failed to restart Nessie container: {e}")
    print("   Manual workaround: Run 'docker compose restart nessie' in your terminal.")

print("\n✨ Clean up complete! All data has been wiped.")


🧹 Starting Data Cleanup Script...
📡 Connected to MinIO at: localhost:9005

🌊 Cleaning up MinIO Data Lake...
   Deleting objects from bucket 'warehouse'...
   ✅ Removed bucket: 'warehouse'
   ✅ Re-created empty bucket: 'warehouse'
   Deleting objects from bucket 'spark-logs'...
   ✅ Removed bucket: 'spark-logs'
   ✅ Re-created empty bucket: 'spark-logs'
   ✅ Created event-logs directory structure in 'spark-logs' bucket.

🏢 Cleaning up ClickHouse Data Warehouse...
   ✅ Dropped ClickHouse table: default.spark_jdbc_test
   ℹ️ ClickHouse database 'gold' does not exist yet (skipping).

🌳 Resetting Nessie Catalog...


 Container nessie Restarting 


   ✅ Nessie Catalog restarted and reset to clean state.

✨ Clean up complete! All data has been wiped.


 Container nessie Started 
